# 01 — Run `mlp3` (learned diffusion times)

Trains `configs/GPS/pcqm4m-subset-GPS+LHKS-K16-MLP3.yaml` for the seeds you list below.

**Prerequisite:** run `00_setup.ipynb` at least once (it creates the Drive folders + downloads the dataset).

**Drive account:** `hert7739@ox.ac.uk`

**Time budget:** ~2 h / seed on A100, ~5 h / seed on T4.

## 1. Configuration — edit the seed list

In [ ]:
# =========================================================
# EDIT: which seeds to run
# =========================================================
SEEDS = [3, 4, 5]
# =========================================================

GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"
CFG             = "configs/GPS/pcqm4m-subset-GPS+LHKS-K16-MLP3.yaml"
OUTSUBDIR       = "mlp_ablation/mlp3"

DRIVE_RESULTS  = f"{DRIVE_WORKSPACE}/results_pcqm4m_subset"
DRIVE_DATASETS = f"{DRIVE_WORKSPACE}/datasets"
print("Will run mlp3 for seeds:", SEEDS)

## 2. GPU check

Prefer A100 or V100; T4 works but is slower.

In [ ]:
!nvidia-smi | head -20

## 3. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone repo + install deps + symlink to Drive

Reinstalls on every fresh Colab VM (the pip env isn't persistent). Dataset and checkpoints DO persist (they're on Drive).

In [ ]:
import os
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
else:
    %cd gdl__2 && !git pull && %cd ..
%cd gdl__2
!bash run_colab/setup.sh
!rm -rf results_pcqm4m_subset datasets
!ln -s {DRIVE_RESULTS}  results_pcqm4m_subset
!ln -s {DRIVE_DATASETS} datasets
!ls -la results_pcqm4m_subset datasets

## 5. WandB login

In [ ]:
import wandb
wandb.login()

## 6. Sanity check — dataset is on Drive

If this errors, go back and run `00_setup.ipynb` first.

In [ ]:
processed = f"{DRIVE_DATASETS}/pcqm4m-v2/processed/geometric_data_processed.pt"
assert os.path.exists(processed), f"Dataset missing at {processed}. Run 00_setup.ipynb first."
sz_gb = os.path.getsize(processed) / 1e9
print(f"OK  geometric_data_processed.pt = {sz_gb:.1f} GB")

## 7. Run training for each seed

Loops sequentially, skips seeds that already have a checkpoint on Drive. Safe to re-run after a disconnect.

In [ ]:
import os, glob, datetime, subprocess

OUTBASE = f"results_pcqm4m_subset/{OUTSUBDIR}"

for seed in SEEDS:
    outdir = f"{OUTBASE}/seed{seed}"
    if glob.glob(f"{outdir}/*/ckpt/*.ckpt"):
        print(f"[skip] mlp3 seed={seed} — checkpoint already exists at {outdir}")
        continue
    os.makedirs(outdir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f">>> mlp3 seed={seed}  start: {datetime.datetime.now().isoformat(timespec='seconds')}")
    print(f">>> cfg:    {CFG}")
    print(f">>> outdir: {outdir}")
    print(f"{'='*60}", flush=True)
    ret = subprocess.call(["python", "main.py", "--cfg", CFG,
                           "--repeat", "1", "seed", str(seed),
                           "out_dir", outdir])
    status = "OK" if ret == 0 else f"FAILED (exit {ret})"
    print(f">>> mlp3 seed={seed}  done:  {datetime.datetime.now().isoformat(timespec='seconds')}  [{status}]")
    if ret != 0:
        raise RuntimeError(f"Training failed for mlp3 seed={seed}")
print("\n=== All mlp3 runs complete ===")

## 8. Verify checkpoints on Drive

In [ ]:
import subprocess
out = subprocess.check_output(
    ["find", f"{DRIVE_RESULTS}/{OUTSUBDIR}", "-name", "*.ckpt"], text=True)
paths = sorted(out.strip().split('\n')) if out.strip() else []
print(f"mlp3 has {len(paths)} checkpoint(s) on Drive:")
for p in paths:
    sz = os.path.getsize(p) / 1e6
    print(f"  {sz:6.1f} MB  {p}")